In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, from_unixtime
import json

# drivers = [
#     "/opt/spark/external-jars/hadoop-aws-3.3.4.jar",
#     "/opt/spark/external-jars/aws-java-sdk-bundle-1.12.262.jar",
#     "/opt/spark/external-jars/wildfly-openssl-1.0.7.Final.jar",
#     "/opt/spark/external-jars/postgresql-42.6.0.jar",
# ]

spark = (SparkSession.builder
         .appName("jupyter-spark")
         .master("spark://spark-master:7077")
         # .config("spark.jars", ",".join(drivers))
         .getOrCreate()
        )

In [ ]:
# Читаем JSON с правильными опциями
df_raw = (spark.read
          .option("multiline", "true")
          .option("encoding", "UTF-8")
          .json("/shared_data/result.json"))

# Разворачиваем структуру
df_chats = (df_raw
            .select(explode("chats.list").alias("chat"))
            .select(
                col("chat.id").alias("chat_id"),
                col("chat.name").alias("chat_name"),
                col("chat.type").alias("chat_type"),
                explode("chat.messages").alias("message")
            )
            .select(
                "chat_id", "chat_name", "chat_type",
                col("message.id").alias("message_id"),
                col("message.date").alias("date"),
                col("message.from").alias("from"),
                col("message.from_id").alias("from_id"),
                col("message.text").alias("text"),
                col("message.type").alias("message_type")
            ))

df_chats.show(10, truncate=False)

In [ ]:
print("Количество сообщений:")
(
    df_chats.select(["chat_name", "date", "from", "text"])
            .filter(col("chat_name") == "Бу - Скелитор")
            .filter(col("text").like("%угар%"))
            .count()
)

In [ ]:
(
    df_chats.select(["chat_name", "date", "from", "text"])
            .filter(col("chat_name") == "Бу - Скелитор")
            .filter(col("text").like("%угар"))
            .show(truncate=100)
)

In [ ]:
spark.stop()